In [ ]:
import pandas as pd
import json
import os

# 1. Asegurar la existencia de la carpeta
os.makedirs("~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data", exist_ok=True)

def construir_grafo_desde_gtfs(ruta_stops, ruta_stop_times):
    print("📊 [1/3] Extrayendo Vértices desde stops.txt...")
    df_stops = pd.read_csv("~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stops.txt", sep=',')

    mis_paraderos = {}
    for _, fila in df_stops.iterrows():
        mis_paraderos[str(fila['stop_id'])] = (float(fila['stop_lat']), float(fila['stop_lon']))

    print("🚌 [2/3] Procesando Aristas desde stop_times.txt (Big Data)...")
    df_times = pd.read_csv("~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stop_times.txt", usecols=['trip_id', 'stop_id', 'stop_sequence'], sep=',')
    df_times = df_times.sort_values(by=['trip_id', 'stop_sequence'])

    trip_ids = df_times['trip_id'].values
    stop_ids = df_times['stop_id'].values

    diccionario_servicios = {}
    adyacencias_grafo = {nodo: set() for nodo in mis_paraderos.keys()}

    print("⚙️ [3/3] Tejiendo la Red Vectorial del Grafo...")
    for i in range(len(trip_ids) - 1):
        viaje_actual = trip_ids[i]
        viaje_siguiente = trip_ids[i + 1]

        if viaje_actual == viaje_siguiente:
            origen = str(stop_ids[i])
            destino = str(stop_ids[i + 1])
            micro = str(viaje_actual).split('-')[0]

            llave_tramo = f"{origen}-{destino}"
            if llave_tramo not in diccionario_servicios:
                diccionario_servicios[llave_tramo] = set()
            diccionario_servicios[llave_tramo].add(micro)

            if origen in adyacencias_grafo:
                adyacencias_grafo[origen].add(destino)

    # Limpieza Final para compatibilidad con JSON
    diccionario_servicios = {k: list(v) for k, v in diccionario_servicios.items()}
    adyacencias_grafo = {k: list(v) for k, v in adyacencias_grafo.items()}

    print(f"✅ ¡GTFS LISTO! {len(mis_paraderos)} paraderos y {len(diccionario_servicios)} calles.")
    return mis_paraderos, diccionario_servicios, adyacencias_grafo

# ==========================================
# EJECUCIÓN Y EXPORTACIÓN A JSON
# ==========================================
ruta_stops = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stops.txt"
ruta_times = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/stop_times.txt"
ruta_salida_gtfs = "~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/gtfs_limpio.json"

paraderos, servicios, conexiones = construir_grafo_desde_gtfs(ruta_stops, ruta_times)

print("💾 Exportando a JSON...")
# Empaquetamos todo el Grafo en un solo archivo JSON
datos_exportar = {
    "paraderos": paraderos,
    "servicios": servicios,
    "conexiones": conexiones
}

with open(ruta_salida_gtfs, "w", encoding="utf-8") as archivo:
    json.dump(datos_exportar, archivo, ensure_ascii=False, indent=4)

print(f"🚀 ¡Listo! Grafo completo guardado en: {"~/sic_2026_c-p_cohort_1/proyectos/Grupo_2/data/gtfs_limpio.json"}")